In [13]:
# check for GPU
import torch
device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [14]:
import pandas as pd
import numpy as np
df = pd.read_csv('fmnist_small.csv')
df.head(3)

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,9,0,0,0,0,0,0,0,0,0,...,0,7,0,50,205,196,213,165,0,0
1,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,1,0,0,0,...,142,142,142,21,0,3,0,0,0,0


In [15]:
from sklearn.model_selection import train_test_split
X = df.drop('label', axis=1)
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [16]:
X_train = X_train/255
X_test = X_test/255

In [17]:
X_train = X_train.to_numpy()
X_test = X_test.to_numpy()
y_train = y_train.to_numpy()
y_test = y_test.to_numpy()

In [18]:
import torch
X_train = torch.from_numpy(X_train).to(torch.float32)
X_test = torch.from_numpy(X_test).to(torch.float32)
y_train = torch.from_numpy(y_train).to(torch.long)
y_test = torch.from_numpy(y_test).to(torch.long)

In [19]:
from torch.utils.data import Dataset, DataLoader
class CustomDataset(Dataset):
  def __init__(self, features, labels):
    self.features = features
    self.labels = labels

  def __len__(self):
    return len(self.features)

  def __getitem__(self, index):
    return self.features[index], self.labels[index]

In [20]:
train_ds = CustomDataset(X_train, y_train)
test_ds = CustomDataset(X_test, y_test)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, pin_memory=True) # Set pin_memory variable to true
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, pin_memory=True)

In [21]:
import torch
import torch.nn as nn

class Neural_Network(nn.Module):
  def __init__(self, num_features):
    super().__init__()
    self.model = nn.Sequential(
        nn.Linear(num_features, 128),
        nn.ReLU(),
        nn.Linear(128, 64),
        nn.ReLU(),
        nn.Linear(64, 10)
    )

  def forward(self, num_features):
    return self.model(num_features)

In [22]:
learning_rate = 0.1
epochs = 100

In [23]:
model = Neural_Network(X_train.shape[1])
model = model.to(device) #set device
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

for epoch in range(epochs):
  for batch_features, batch_labels in train_loader:
    # move data to gpu
    batch_features = batch_features.to(device)
    batch_labels = batch_labels.to(device)

    y_pred = model(batch_features)
    loss = loss_function(y_pred, batch_labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

  print(f'Epoch: {epoch+1}, Loss:{loss.item()}')

Epoch: 1, Loss:0.6381089687347412
Epoch: 2, Loss:0.8898695111274719
Epoch: 3, Loss:0.45780283212661743
Epoch: 4, Loss:0.627249002456665
Epoch: 5, Loss:0.7062526941299438
Epoch: 6, Loss:0.5308444499969482
Epoch: 7, Loss:0.3226824700832367
Epoch: 8, Loss:0.44073063135147095
Epoch: 9, Loss:0.41915374994277954
Epoch: 10, Loss:0.3186505138874054
Epoch: 11, Loss:0.3491136431694031
Epoch: 12, Loss:0.4952065944671631
Epoch: 13, Loss:0.5653367042541504
Epoch: 14, Loss:0.19030791521072388
Epoch: 15, Loss:0.21003803610801697
Epoch: 16, Loss:0.2946736812591553
Epoch: 17, Loss:0.3762250542640686
Epoch: 18, Loss:0.17232199013233185
Epoch: 19, Loss:0.14371046423912048
Epoch: 20, Loss:0.4370831847190857
Epoch: 21, Loss:0.23507750034332275
Epoch: 22, Loss:0.23568110167980194
Epoch: 23, Loss:0.4079216718673706
Epoch: 24, Loss:0.38012605905532837
Epoch: 25, Loss:0.40354621410369873
Epoch: 26, Loss:0.2953886389732361
Epoch: 27, Loss:0.1391836255788803
Epoch: 28, Loss:0.19416803121566772
Epoch: 29, Loss:0.

In [24]:
model.eval()
total = 0
correct = 0

with torch.no_grad():
  for batch_features, batch_labels in test_loader:
    # move data to gpu
    batch_features = batch_features.to(device)
    batch_labels = batch_labels.to(device)

    y_pred = model(batch_features)
    _, predicted = torch.max(y_pred, 1)
    total += batch_labels.shape[0]
    correct += (predicted == batch_labels).sum().item()
print(f'Accuracy: {correct/total}')

Accuracy: 0.85
